# Zenoh — key expressions, subscriptions and queries

Two sessions share an in-process `VirtualZenohNetwork`, so nothing here needs the native library or a network. To talk to real peers, drop `UseVirtual(net)` and use `Connect("tcp/host:7447")` or `Listen("tcp/0.0.0.0:7447")`; the native `iotcom_zenoh` library ships with the package.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.Zenoh, 0.22.0-preview.1"

## Key expressions
Keys are slash-separated chunks. `*` matches one chunk, `**` any number of chunks (including none), `$*` any part of a chunk.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Adapters.Zenoh;

foreach (var (pattern, key) in new[] { ("plant/*/temp", "plant/line1/temp"), ("plant/*/temp", "plant/a/b/temp"), ("plant/**", "plant/a/b/c"), ("sensor$*/temp", "sensor42/temp") })
    Console.WriteLine($"{pattern,-14} {(ZenohKeyExpr.Includes(pattern, key) ? "matches    " : "no match   ")} {key}");
Console.WriteLine($"plant/*/temp and plant/line1/** can both match one key: {ZenohKeyExpr.Intersects("plant/*/temp", "plant/line1/**")}");
Console.WriteLine($"valid \"plant//x\": {ZenohKeyExpr.IsValid("plant//x")}");

## Put, subscribe and delete
A subscription gets every put and delete whose key matches its expression, from other sessions.

In [ ]:
var net = new VirtualZenohNetwork();
var gateway = ZenohSession.Create(o => o.UseVirtual(net));
var sensor = ZenohSession.Create(o => o.UseVirtual(net));
await gateway.ConnectAsync();
await sensor.ConnectAsync();

var seen = new List<string>();
var subscription = gateway.Subscribe("plant/*/temp", s => { lock (seen) seen.Add($"{s.Kind} {s.Key} {s.Text}".TrimEnd()); });
await sensor.PutAsync("plant/line1/temp", "21.5");
await sensor.PutAsync("plant/line2/temp", "22.1");
await sensor.PutAsync("plant/line1/pressure", "1.8");   // other key: not delivered
await sensor.DeleteAsync("plant/line1/temp");
await Task.Delay(200);
lock (seen) foreach (var line in seen) Console.WriteLine(line);
subscription.Dispose();

## Watching as a stream
`WatchAsync` turns a key expression into an `IAsyncEnumerable`.

In [ ]:
var cts = new CancellationTokenSource(TimeSpan.FromSeconds(10));
var watcher = Task.Run(async () =>
{
    var keys = new List<string>();
    await foreach (var s in gateway.WatchAsync("plant/**", cts.Token))
    {
        keys.Add(s.Key);
        if (keys.Count == 3) break;
    }
    return keys;
});
// The subscription is declared when enumeration starts, so keep publishing until the watcher has three samples.
for (var i = 0; i < 200 && !watcher.IsCompleted; i++) { await sensor.PutAsync("plant/line1/temp", $"v{i}"); await Task.Delay(10); }
Console.WriteLine(string.Join(", ", await watcher));

## Queryables and get
A queryable answers `get` requests; `GetAsync` collects the replies of every matching queryable, including error replies.

In [ ]:
var info = sensor.DeclareQueryable("plant/*/info", async q =>
{
    await q.ReplyAsync("plant/line1/info", $"state=running query={q.Parameters}");
    await q.ReplyErrorAsync("line2 offline");
});
foreach (var reply in await gateway.GetAsync("plant/*/info?detail=1"))
    Console.WriteLine(reply.IsError ? $"error: {reply.ErrorText}" : $"{reply.Sample!.Key}: {reply.Sample.Text}");
info.Dispose();

## Read-only sessions
Put, delete and replies change what other nodes see. A session with `ReadOnly = true` refuses them but still subscribes and queries.

In [ ]:
var viewer = ZenohSession.Create(o => { o.UseVirtual(net); o.ReadOnly = true; });
await viewer.ConnectAsync();
try { await viewer.PutAsync("plant/line1/setpoint", "99"); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
Console.WriteLine($"get still works: {(await viewer.GetAsync("plant/*/info", timeout: TimeSpan.FromMilliseconds(500))).Count} replies");
await viewer.DisposeAsync();
await sensor.DisposeAsync();
await gateway.DisposeAsync();

## Going further
`iotcom zenoh sub "plant/**" --sim`, `iotcom zenoh get "plant/*/info" --sim`, `iotcom zenoh pub plant/line1/setpoint 42 --sim --allow-write`. See `docs/en/protocols/zenoh.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*